# 5주차 실습. DESeq2로 차등발현 분석하기

이 파일을 직접 고치지 말고 먼저 본인 폴더로 복사한다. Qoka 대화 입력창에 입력한다.

> course/labs/week5/week5_deseq2.ipynb 를 analysis/week5/ 폴더로 복사해서 열어줘.

열리면 오른쪽 위 커널을 **Qoka Run Environment**로 고른다.

## 이 노트북을 쓰는 방법

- 튜토리얼 "A simple PyDESeq2 workflow"의 단계를 셀 하나씩 나눔. 코드는 미리 적혀 있음
- 셀마다 다음 순서 (3주차와 같음)
  1. **프롬프트**를 읽는다. 에이전트에게 이 셀을 시킨다면 할 말
  2. 코드와 주석을 읽고, 프롬프트의 말이 어느 줄이 되었는지 찾는다
  3. 실행하기 전에 **예상** 칸에 적는다
  4. 실행하고 결과를 예상과 비교한다. 결과 설명은 강의노트
- 셀은 위에서부터 순서대로 실행

| 실습 | 셀 | 내용 | 강의노트 |
|---|---|---|---|
| 1 | 1–8 | 데이터, `~dex` 모형, 검정 | 5.3–5.6 |
| 2 | 9–12 | VST, PCA, `~cell + dex` | 5.7–5.8 |
| 3 | 13–16 (+17 선택) | shrinkage, 결과 해석 | 5.9–5.10 |

## 준비

패키지를 설치하고 파일 위치를 찾는다. 그대로 실행한다. "restart the kernel"이 나오면 커널 재시작 후 처음부터.

In [ ]:
%pip install -q pydeseq2==0.5.2 matplotlib mygene gseapy

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pydeseq2.dds import DeseqDataSet   # 모형: size factor, dispersion, 배수
from pydeseq2.ds import DeseqStats      # 검정: Wald test, padj

# data/ 폴더가 나올 때까지 위로 올라가며 프로젝트 폴더를 찾음 (3주차와 같음)
root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent
out_dir = root / "results" / "week5"          # 그림과 표를 저장할 곳
out_dir.mkdir(parents=True, exist_ok=True)

# 자주 보는 유전자의 Ensembl ID
GENES = {"FKBP5": "ENSG00000096060", "TSC22D3": "ENSG00000157514", "PER1": "ENSG00000179094",
         "ZBTB16": "ENSG00000109906", "TARDBP": "ENSG00000120948", "ANGPTL7": "ENSG00000171819"}
print((root / "data" / "airway_scaledcounts.subset.tsv").exists())

---
# 실습 1. 데이터, `~dex` 모형, 검정

## 셀 1. 카운트 읽기

**프롬프트**: > 카운트 파일을 튜토리얼 모양(행: 샘플, 열: 유전자)으로 읽어줘.

**예상**: 표의 행과 열은 몇 개일까? (3주차: 38,694 × 4)

> (실행하기 전에 여기에 적는다)

In [ ]:
# 소수점이 쉼표인 파일 (3주차 CLAUDE.md 규칙)
counts = pd.read_csv(root / "data" / "airway_scaledcounts.subset.tsv", sep="\t", decimal=",", index_col="ensgene")

# .T: 행과 열을 뒤집음. astype(int): 정수로
counts_df = counts.T.astype(int)
counts_df.shape

## 셀 2. 샘플 정보와 기준 수준

**프롬프트**: > 샘플 정보를 읽고 dex의 기준 수준을 untrt로 정해줘.

**예상**: `dex` 수준 순서의 첫 번째는? 이 줄이 없으면? (강의노트 5.9절)

> (실행하기 전에 여기에 적는다)

In [ ]:
metadata = pd.read_csv(root / "data" / "week4" / "samples.tsv", sep="\t", index_col="sample")
metadata = metadata.loc[counts_df.index, ["cell", "dex"]]   # 카운트와 같은 샘플 순서

# 기준 수준(비교의 분모) = 첫 번째 값 untrt
metadata["dex"] = pd.Categorical(metadata["dex"], categories=["untrt", "trt"])
print(list(metadata["dex"].cat.categories))
metadata

## 셀 3. 유전자 거르기

**프롬프트**: > 튜토리얼과 같은 기준으로 유전자를 걸러줘.

**예상**: 몇 개가 남을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 네 샘플 합이 10 이상인 유전자만 남김 (튜토리얼과 같은 기준)
counts_df = counts_df.loc[:, counts_df.sum(axis=0) >= 10]
counts_df.shape

## 셀 4. `~dex` 모형 맞추기

**프롬프트**: > 설계식 ~dex로 DESeq2 모형을 맞춰줘.

**예상**: 계수 이름은? (강의노트 5.9절 "계수 이름 읽기")

> (실행하기 전에 여기에 적는다)

In [ ]:
with warnings.catch_warnings(record=True) as w:     # 경고를 모아 둠 (셀 6에서 봄)
    warnings.simplefilter("always")
    dds = DeseqDataSet(counts=counts_df, metadata=metadata, design="~dex", quiet=True)
    dds.deseq2()   # size factor → dispersion → 배수를 한 번에

list(dds.varm["LFC"].columns)   # 계수 이름

## 셀 5. Size factor

**프롬프트**: > 샘플별 size factor와 TARDBP의 보정 전후 값을 보여줘.

**예상**: size factor가 가장 작은 샘플은? TARDBP 2번 쌍(3,408 → 1,603)은 보정 후에도 절반일까? (강의노트 5.4절)

> (실행하기 전에 여기에 적는다)

In [ ]:
sf = dds.obs["size_factors"]      # 샘플별 size factor (median-of-ratios)
g = GENES["TARDBP"]
pd.DataFrame({"size factor": sf.round(3),
              "TARDBP 원래 값": counts_df[g],
              "TARDBP 보정 값": (counts_df[g] / sf).round(0)})   # 보정 = count / size factor

## 셀 6. Dispersion과 경고

**프롬프트**: > dispersion 중앙값과 경고 메시지를 보여줘.

**예상**: ① 자기 count로 잰 값과 ④ 최종값 중 어느 쪽이 클까? (강의노트 5.5절)

> (실행하기 전에 여기에 적는다)

In [ ]:
print("① 자기 count로 잰 값:", round(dds.var["genewise_dispersions"].median(), 3))
print("② 기준선:          ", round(dds.var["fitted_dispersions"].median(), 3))
print("④ 최종:            ", round(dds.var["dispersions"].median(), 3))

# 의미 있는 경고는 UserWarning. FutureWarning은 라이브러리 내부 경고
for msg in {str(x.message) for x in w if x.category is UserWarning}:
    print("경고:", msg)

## 셀 7. 검정

**프롬프트**: > trt를 untrt와 비교해서 padj 0.05 미만인 유전자를 세줘.

**예상**: p < 0.05와 padj < 0.05 중 어느 쪽이 많을까? (강의노트 5.6절)

> (실행하기 전에 여기에 적는다)

In [ ]:
# contrast = [열, 분자, 분모] → log2FoldChange = log2(trt / untrt)
ds = DeseqStats(dds, contrast=["dex", "trt", "untrt"], quiet=True)
ds.summary()
res = ds.results_df.copy()

sig = res["padj"] < 0.05
print("p < 0.05:   ", (res["pvalue"] < 0.05).sum())
print("padj < 0.05:", sig.sum(), "(증가", (sig & (res.log2FoldChange > 0)).sum(), "/ 감소", (sig & (res.log2FoldChange < 0)).sum(), ")")
print("padj NaN:   ", res["padj"].isna().sum())   # count가 낮아 검정에서 빠진 유전자

## 셀 8. Positive control

**프롬프트**: > FKBP5, TSC22D3, PER1, ZBTB16, TARDBP의 결과를 보여줘.

**예상**: 앞의 네 개(glucocorticoid 반응 유전자)의 부호는? TARDBP는 유의할까?

> (실행하기 전에 여기에 적는다)

In [ ]:
names = ["FKBP5", "TSC22D3", "PER1", "ZBTB16", "TARDBP"]
table = res.loc[[GENES[n] for n in names], ["baseMean", "log2FoldChange", "padj"]].set_axis(names)
table["padj"] = table["padj"].map("{:.1e}".format)   # 아주 작은 수는 지수 표기로
table.round(2)

---
# 실습 2. 탐색과 세포주를 넣은 설계

## 셀 9. VST와 PCA

**프롬프트**: > rnaseqGene 예제처럼 VST 후 유전자 500개로 PCA를 그려줘.

**예상**: PC1은 세포주와 처리 여부 중 무엇으로 나뉜까? (강의노트 5.7절)

> (실행하기 전에 여기에 적는다)

In [ ]:
dds.vst(use_design=True)   # rnaseqGene의 vst(dds, blind = FALSE)
vst = pd.DataFrame(dds.layers["vst_counts"], index=counts_df.index, columns=counts_df.columns)

# 샘플 간 차이가 큰 유전자 500개 → 유전자별 평균을 빼고 PCA
top = vst.var().sort_values(ascending=False).index[:500]
X = vst[top] - vst[top].mean()
U, S, _ = np.linalg.svd(X.values, full_matrices=False)
pcs = pd.DataFrame(U[:, :2] * S[:2], index=vst.index, columns=["PC1", "PC2"])
ratio = S**2 / (S**2).sum()   # 각 축의 설명 비율

fig, ax = plt.subplots(figsize=(5, 4))
for s, row in pcs.iterrows():
    color = "tab:blue" if metadata.loc[s, "dex"] == "trt" else "gray"     # 색: 처리 여부
    marker = "o" if metadata.loc[s, "cell"] == "N61311" else "s"          # 모양: 세포주
    ax.scatter(row.PC1, row.PC2, s=90, color=color, marker=marker)
    ax.annotate(s, (row.PC1, row.PC2), xytext=(6, 6), textcoords="offset points")
ax.set_xlabel(f"PC1 ({ratio[0]:.0%})")
ax.set_ylabel(f"PC2 ({ratio[1]:.0%})")
plt.show()

## 셀 10. 샘플 사이 거리

**프롬프트**: > 샘플 사이 거리를 표로 보여줘.

**예상**: 가장 먼 두 샘플은?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 전체 유전자의 VST 값으로 계산한 유클리드 거리
d = np.sqrt(((vst.values[:, None, :] - vst.values[None, :, :]) ** 2).sum(axis=2))
pd.DataFrame(d, index=vst.index, columns=vst.index).round(1)

## 셀 11. `~cell + dex` 모형

**프롬프트**: > 설계식 ~cell + dex로 다시 맞추고 ~dex와 유의한 유전자 수를 비교해줘.

**예상**: 세포주를 넣으면 유의한 유전자는 늘까 줄까? (강의노트 5.8절)

> (실행하기 전에 여기에 적는다)

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")        # 경고는 셀 6과 같음
    dds_cell = DeseqDataSet(counts=counts_df, metadata=metadata, design="~cell + dex", quiet=True)
    dds_cell.deseq2()
    ds_cell = DeseqStats(dds_cell, contrast=["dex", "trt", "untrt"], quiet=True)
    ds_cell.summary()
res_cell = ds_cell.results_df.copy()       # ~dex 결과(res)는 그대로 둠

print("계수 이름:", list(dds_cell.varm["LFC"].columns))
print("padj < 0.05  ~dex:", (res.padj < 0.05).sum(), " ~cell + dex:", (res_cell.padj < 0.05).sum())
print("두 설계 모두 유의:", ((res.padj < 0.05) & (res_cell.padj < 0.05)).sum())

## 셀 12. ANGPTL7, 두 설계 비교

**프롬프트**: > ANGPTL7의 log2FoldChange, lfcSE, padj를 두 설계에서 나란히 보여줘.

**예상**: 두 설계에서 크게 달라지는 것은 배수일까, 표준오차일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
g = GENES["ANGPTL7"]
cols = ["log2FoldChange", "lfcSE", "padj"]
pd.DataFrame({"~dex": res.loc[g, cols], "~cell + dex": res_cell.loc[g, cols]}).T

---
# 실습 3. Shrinkage와 결과 해석

## 셀 13. LFC shrinkage

**프롬프트**: > ~cell + dex 모형의 dex 효과에 lfc_shrink를 적용해줘.

**예상**: FKBP5와 ANGPTL7 중 더 많이 줄어드는 것은? (강의노트 5.9절)

> (실행하기 전에 여기에 적는다)

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    # 계수 이름으로 지정. 기준 수준이 untrt이므로 dex[T.trt] = trt / untrt
    ds_cell.lfc_shrink(coeff="dex[T.trt]")
res_shrunk = ds_cell.results_df.copy()   # log2FoldChange, lfcSE만 바뀜. padj는 그대로

names = ["FKBP5", "TSC22D3", "ANGPTL7"]
ids = [GENES[n] for n in names]
pd.DataFrame({"shrink 전": res_cell.loc[ids, "log2FoldChange"].values,
              "shrink 후": res_shrunk.loc[ids, "log2FoldChange"].values}, index=names).round(2)

## 셀 14. MA plot

**프롬프트**: > MA plot을 그려줘.

**예상**: 왼쪽(count가 낮은 쪽) 점들은 어디에 모일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 가로: baseMean, 세로: shrink 후 log2FoldChange. 색이 있는 점: padj < 0.05
ds_cell.plot_MA(s=4)

## 셀 15. 유전자 이름과 volcano plot

**프롬프트**: > 유전자 이름을 붙이고 volcano plot을 그려줘. 기준선은 padj 0.05, |log2FoldChange| 1.

**예상**: padj 1위 유전자는 가장 크게 변한 유전자일까? (강의노트 5.10절)

> (실행하기 전에 여기에 적는다)

In [ ]:
import mygene

# 유의한 유전자의 Ensembl ID → 유전자 이름 (온라인 조회, 인터넷 필요)
sig_ids = list(res_shrunk.index[res_shrunk.padj < 0.05])
q = mygene.MyGeneInfo().querymany(sig_ids, scopes="ensembl.gene", fields="symbol",
                                  species="human", as_dataframe=True, verbose=False)
symbols = q["symbol"].dropna()
symbols = symbols[~symbols.index.duplicated()]
res_shrunk["symbol"] = symbols.reindex(res_shrunk.index)
res_shrunk.sort_values("padj").to_csv(out_dir / "deseq2_results.csv")   # 결과 표 저장

up = (res_shrunk.padj < 0.05) & (res_shrunk.log2FoldChange > 1)       # 증가
down = (res_shrunk.padj < 0.05) & (res_shrunk.log2FoldChange < -1)    # 감소
print("이름이 붙은 유전자:", len(symbols), "/", len(sig_ids))
print("증가:", up.sum(), " 감소:", down.sum())

y = -np.log10(res_shrunk.padj.clip(lower=1e-150))   # 아주 작은 padj는 1e-150에서 자름
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(res_shrunk.log2FoldChange, y, s=3, color="lightgray")
ax.scatter(res_shrunk.log2FoldChange[up | down], y[up | down], s=4, color="tab:blue")
ax.axhline(-np.log10(0.05), color="tab:orange", linestyle="--")
ax.axvline(-1, color="tab:orange", linestyle="--")
ax.axvline(1, color="tab:orange", linestyle="--")
ax.set_xlabel("log2FoldChange (shrunken)")
ax.set_ylabel("-log10(padj)")
plt.show()

res_shrunk.sort_values("padj")[["symbol", "baseMean", "log2FoldChange", "padj"]].head(5)

## 셀 16. Heatmap

**프롬프트**: > padj 상위 20개 유전자의 heatmap을 그려줘.

**예상**: 처리군 두 열의 색은 같을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
top20 = res_shrunk.sort_values("padj").index[:20]
order = ["1_control", "2_control", "1_treated", "2_treated"]   # 대조군 2, 처리군 2
z = vst.loc[order, top20].T
z = z.sub(z.mean(axis=1), axis=0)          # 유전자마다 평균을 뺌: 빨강 = 평균보다 높음

fig, ax = plt.subplots(figsize=(4, 6))
im = ax.imshow(z.values, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
ax.set_xticks(range(4), order, rotation=30)
ax.set_yticks(range(20), [symbols.get(g, g) for g in top20], fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.6)
plt.show()

## 셀 17. (선택) Enrichment

**프롬프트**: > 증가 유전자로 Enrichr MSigDB Hallmark 분석을 해줘.

**예상**: 1위 term의 이름대로 해석해도 될까? 겹친 유전자를 본다 (강의노트 5.10절)

> (실행하기 전에 여기에 적는다)

In [ ]:
import gseapy as gp

up_genes = sorted(set(res_shrunk.loc[up, "symbol"].dropna()))   # 증가 유전자 이름
enr = gp.enrichr(gene_list=up_genes, gene_sets=["MSigDB_Hallmark_2020"], organism="human", outdir=None)
print("입력 유전자:", len(up_genes))
enr.results.sort_values("Adjusted P-value")[["Term", "Overlap", "Adjusted P-value", "Genes"]].head(5)